# 10.18 — Neural style transfer
Style transfer optimizes an image to keep content features while matching style-feature correlations. Gram matrices encode texture-like statistics, and a content/style weighted loss controls the result. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

SEED = 1014
rng = np.random.default_rng(SEED)


def normalize01(x):
    arr = np.asarray(x, dtype=float)
    lo = float(np.min(arr))
    hi = float(np.max(arr))
    span = hi - lo
    if span < 1e-12:
        return np.zeros_like(arr)
    return (arr - lo) / span


def synthetic_faces(n=64, size=16, seed=SEED):
    local_rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[-1:1:complex(size), -1:1:complex(size)]
    faces = []
    for i in range(n):
        face = 0.25 + 0.25 * np.exp(-2.0 * (xx ** 2 + yy ** 2))
        eye_y = -0.30 + local_rng.normal(0.0, 0.03)
        eye_dx = 0.35 + local_rng.normal(0.0, 0.02)
        mouth_y = 0.45 + local_rng.normal(0.0, 0.04)
        face -= 0.35 * np.exp(-90.0 * ((xx - eye_dx) ** 2 + (yy - eye_y) ** 2))
        face -= 0.35 * np.exp(-90.0 * ((xx + eye_dx) ** 2 + (yy - eye_y) ** 2))
        face += 0.14 * np.exp(-70.0 * (xx ** 2 + (yy - 0.06) ** 2))
        face -= 0.20 * np.exp(-55.0 * (xx ** 2 + (yy - mouth_y) ** 2))
        face += local_rng.normal(0.0, 0.025, size=(size, size))
        faces.append(normalize01(face))
    return np.asarray(faces).reshape(n, size * size)


def load_tiny_faces():
    try:
        from sklearn.datasets import fetch_olivetti_faces
        faces = fetch_olivetti_faces(download_if_missing=False, shuffle=True, random_state=SEED)
        data = faces.data[:64]
        return data, (64, 64), "Olivetti faces cache"
    except Exception:
        return synthetic_faces(), (16, 16), "synthetic face fallback"


def make_f9_ladder(seed=SEED):
    local_rng = np.random.default_rng(seed)
    d1 = local_rng.normal(0.0, 1.0, size=(80, 1))
    d2, _ = make_moons(n_samples=120, noise=0.08, random_state=seed)
    d3, _ = make_blobs(n_samples=144, centers=4, n_features=6, cluster_std=0.65, random_state=seed)
    digits = load_digits()
    d4 = digits.data[:80] / 16.0
    faces, face_shape, face_source = load_tiny_faces()
    ladder = [
        {"name": "D1 gaussian", "data": d1, "shape": (1, 1), "kind": "vector"},
        {"name": "D2 two moons", "data": d2, "shape": (1, 2), "kind": "points"},
        {"name": "D3 mixture", "data": d3, "shape": (2, 3), "kind": "vector"},
        {"name": "D4 sklearn digits", "data": d4, "shape": (8, 8), "kind": "image"},
        {"name": "D5 faces (" + face_source + ")", "data": faces, "shape": face_shape, "kind": "image"},
    ]
    return ladder


def centered_scaled(data):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(np.asarray(data, dtype=float))
    return scaled, scaler


def choose_components(data, cap=8):
    n_samples, n_features = data.shape
    return max(1, min(cap, n_samples - 1, n_features))


def pca_reconstruct(data, n_components):
    n_components = max(1, min(n_components, data.shape[0] - 1, data.shape[1]))
    pca = PCA(n_components=n_components, random_state=SEED)
    z = pca.fit_transform(data)
    recon = pca.inverse_transform(z)
    return recon, z, pca


def mse(a, b):
    return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))


def two_sample_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n = min(len(a), len(b), 80)
    a = a[:n]
    b = b[:n]
    aa = pairwise_distances(a, a).mean()
    bb = pairwise_distances(b, b).mean()
    ab = pairwise_distances(a, b).mean()
    return float(2.0 * ab - aa - bb)


def preview_array(ax, sample, shape, title):
    arr = np.asarray(sample)
    if int(np.prod(shape)) == arr.size and shape[0] > 1 and shape[1] > 1:
        ax.imshow(arr.reshape(shape), cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        ax.plot(arr.ravel(), marker="o")
    ax.set_title(title, fontsize=9)


def show_ladder_preview(ladder):
    fig, axes = plt.subplots(1, len(ladder), figsize=(14, 2.8))
    for ax, rung in zip(axes, ladder):
        preview_array(ax, rung["data"][0], rung["shape"], rung["name"])
    plt.tight_layout()
    return fig


## The concept, built once (D1)
For feature matrix $F\in\mathbb{R}^{C	imes N}$, neural style transfer uses the Gram matrix $G=FF^	op/N$ and combines losses as $L=lpha L_{content}+eta L_{style}$. The lesson matrix $F=egin{bmatrix}1&2\3&4\end{bmatrix}$ gives $G_{11}=2.500$, $G_{12}=5.500$, and $G_{22}=12.50$.

In [ ]:

def gram_style_loss(features, style_features, alpha=1.0, beta=2.0):
    features = np.asarray(features, dtype=float)
    style_features = np.asarray(style_features, dtype=float)
    gram = features @ features.T / features.shape[1]
    style_gram = style_features @ style_features.T / style_features.shape[1]
    content_loss = float(np.mean((features - style_features) ** 2))
    style_loss = float(np.mean((gram - style_gram) ** 2))
    total = alpha * content_loss + beta * style_loss
    return gram, total, content_loss, style_loss

F = np.array([[1.0, 2.0], [3.0, 4.0]])
G, total_loss, content_loss, style_loss = gram_style_loss(F, F, alpha=1.0, beta=2.0)
assert round(float(G[0, 0]), 3) == 2.500
assert round(float(G[0, 1]), 3) == 5.500
assert round(float(G[1, 1]), 2) == 12.50
assert round(total_loss, 3) == 0.000
print("Gram", np.round(G, 3))


The notebook uses small feature matrices instead of downloaded CNN features. The optimization is still the real NST objective: update generated features to reduce content loss while matching the style Gram matrix.

In [ ]:

def feature_matrix(data_row, channels=4):
    x = np.asarray(data_row, dtype=float).ravel()
    parts = np.array_split(x, channels)
    width = min(len(part) for part in parts)
    trimmed = [part[:width] for part in parts]
    return np.vstack(trimmed)


def optimize_style_features(content_row, style_row, alpha=1.0, beta=4.0, steps=80, lr=0.05):
    content = feature_matrix(content_row)
    style = feature_matrix(style_row)
    generated = content.copy()
    style_gram = style @ style.T / style.shape[1]
    for step in range(steps):
        gram = generated @ generated.T / generated.shape[1]
        content_grad = 2.0 * alpha * (generated - content) / generated.size
        style_grad = 4.0 * beta * ((gram - style_gram) @ generated) / (generated.shape[1] * generated.size)
        generated = generated - lr * (content_grad + style_grad)
    gram = generated @ generated.T / generated.shape[1]
    content_loss = float(np.mean((generated - content) ** 2))
    style_loss = float(np.mean((gram - style_gram) ** 2))
    total = alpha * content_loss + beta * style_loss
    flat = np.resize(generated.ravel(), np.asarray(content_row).size)
    return normalize01(flat), total, content_loss, style_loss


## The dataset ladder (D1–D5)
We use the same F9 ladder inline for every topic: a one-dimensional Gaussian, two moons, a mixture, tiny sklearn digits, and a face rung. The face rung uses a local Olivetti cache when present and otherwise a deterministic no-download synthetic face fallback.

In [ ]:

ladder = make_f9_ladder()
for index, rung in enumerate(ladder, start=1):
    data = rung["data"]
    print(f"D{index}: {rung['name']} | shape={data.shape} | sample_shape={rung['shape']} | kind={rung['kind']}")
show_ladder_preview(ladder)


## Run the SAME method across D1-D5
The same reusable method is applied to every rung and one plan metric is collected per rung.

In [ ]:

results = []
for level, rung in enumerate(ladder, start=1):
    data = normalize01(rung["data"])
    content = data[0]
    style = data[min(1, len(data) - 1)]
    generated, metric, content_loss, style_loss = optimize_style_features(content, style, alpha=1.0, beta=3.0)
    results.append({"level": level, "name": rung["name"], "metric": metric, "sample": generated, "shape": rung["shape"]})
    print(f"D{level} {rung['name']}: content-plus-style error={metric:.5f}, content={content_loss:.5f}, style={style_loss:.5f}")


## Results visualization
The closing figure has generated-sample panels for D1-D5 plus the requested metric curve.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(14, 2.8))
for ax, result in zip(axes, results):
    preview_array(ax, result["sample"], result["shape"], result["name"])
plt.suptitle("Generated or reconstructed samples by rung")
plt.tight_layout()

plt.figure(figsize=(6, 3.2))
plt.plot([r["level"] for r in results], [r["metric"] for r in results], marker="o")
plt.xticks([r["level"] for r in results], [r["name"].split()[0] for r in results])
plt.ylabel("content-plus-style error")
plt.xlabel("F9 rung")
plt.title("Gram style transfer: metric vs. complexity")
plt.grid(True, alpha=0.3)
plt.tight_layout()


## Pitfall on D5: beta too high or pixel loss for style
The lesson warns that style is not pixel loss and that too much $eta$ can overwhelm content activations. We reproduce a content-washed D5 face with high style weight, then rebalance content and Gram-style losses.

In [ ]:

d5 = ladder[-1]
data = normalize01(d5["data"])
content = data[0]
style = data[1]
washed, washed_metric, washed_content, washed_style = optimize_style_features(content, style, alpha=0.2, beta=40.0, steps=100, lr=0.04)
balanced, balanced_metric, balanced_content, balanced_style = optimize_style_features(content, style, alpha=1.0, beta=4.0, steps=100, lr=0.04)
print(f"too-high beta error={washed_metric:.5f}, content={washed_content:.5f}")
print(f"balanced error={balanced_metric:.5f}, content={balanced_content:.5f}")
fig, axes = plt.subplots(1, 4, figsize=(9, 2.4))
preview_array(axes[0], content, d5["shape"], "content")
preview_array(axes[1], style, d5["shape"], "style")
preview_array(axes[2], washed, d5["shape"], "beta too high")
preview_array(axes[3], balanced, d5["shape"], "balanced")
plt.tight_layout()


## Evaluate it + Practice
- Track `content-plus-style error` against a no-skill baseline such as random samples, unconditioned reconstruction, or independent frames.
- Run a cheap sanity check: dimensions match, finite metrics, and generated samples stay in the training range.
- Ablation: replace Gram-style loss with pixel loss or set beta so high that content disappears; the metric should get worse.
- Failure signals: D5 looks plausible in one panel but the metric curve jumps, indicating artifacts hidden by small previews.

Practice 1: change the latent or conditioning dimension and predict which rung changes most.


Practice 2: change the feature split from 4 channels to 2 channels and inspect the style scale.

Practice 3: sweep beta and plot content loss against style loss.